# Phase 3 Exp 2: 長期学習(epochs=100)

## 仮説

Exp 1(`02_exp1_highres_training.ipynb`)で imgsz=1024 によって mAP@0.5 が 0.815 → 0.903 と大幅改善した。しかし学習曲線(val/mAP50)を見ると、50 エポック時点でもまだわずかに上昇傾向にあり、完全には飽和していない。

**仮説**: Exp 1 と同条件で epochs を 50 → 100 に増やせば、特に shower や staircase などまだ伸びしろのあるクラスでさらなる改善が見られる。ただし door / window のような既に上限近いクラスはほとんど変わらないと予想。

## 実験設計(Exp 1 との公正な比較のため、epochs 以外は完全同一)

| 項目 | Exp 1 | Exp 2 | 変更 |
|---|---|---|---|
| Model | yolov8n.pt | yolov8n.pt | 同じ |
| imgsz | 1024 | 1024 | 同じ |
| **epochs** | **50** | **100** | ⭐ 変更点 |
| batch | 8 | 8 | 同じ |
| optimizer | auto (AdamW) | auto (AdamW) | 同じ |
| seed | 42 | 42 | 同じ |
| patience | 15 | 25 | epochs 倍増に合わせて延長 |

## 予想される副作用
- 学習時間が約2倍に
- 過学習リスクが上昇(train でしか良くならない可能性)
- patience を 15→25 にすることで、停滞時の早期終了は維持しつつ epochs 増を活かす

## Section 1: 環境セットアップ

In [ ]:
!nvidia-smi

In [ ]:
import os

WORKDIR = "/content/floor-plan-recognition"

if not os.path.exists(WORKDIR):
    !git clone https://github.com/Mao925/floor-plan-recognition.git {WORKDIR}
else:
    %cd {WORKDIR}
    !git pull

%cd {WORKDIR}
!pwd

In [ ]:
!pip install -q ultralytics roboflow python-dotenv

In [ ]:
import torch
import ultralytics

print(f"PyTorch:        {torch.__version__}")
print(f"Ultralytics:    {ultralytics.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:            {torch.cuda.get_device_name(0)}")
    print(f"VRAM:           {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Section 2: データ準備

In [ ]:
from google.colab import userdata

try:
    ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
    print(f"✅ API キー取得成功: {ROBOFLOW_API_KEY[:3]}***{ROBOFLOW_API_KEY[-3:]}")
except Exception as e:
    print(f"❌ エラー: {e}")

In [ ]:
with open('.env', 'w') as f:
    f.write(f'ROBOFLOW_API_KEY={ROBOFLOW_API_KEY}\n')

!python scripts/download_roboflow.py

In [ ]:
!python scripts/prepare_dataset.py

## Section 3: 学習(epochs=100)

**所要時間予想**: T4 GPU で約 20〜40 分(Exp 1 の2倍)

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data='data/floorplan_yolo/data.yaml',
    epochs=100,             # ⭐ 変更点: 50 → 100
    imgsz=1024,             # Exp 1 と同じ
    batch=8,                # Exp 1 と同じ
    name='exp2_long_yolov8n',
    project='runs/detect',
    patience=25,            # epochs 倍増に合わせて延長
    save=True,
    plots=True,
    device=0,
    seed=42,
)

print("\n✅ 学習完了")

In [ ]:
!find runs -name 'best.pt' | head -5

## Section 4: 評価と三者比較(Baseline vs Exp 1 vs Exp 2)

In [ ]:
from pathlib import Path

candidates = list(Path('runs').rglob('exp2_long_yolov8n/weights/best.pt'))
assert candidates, "best.pt が見つかりません"
best_pt = candidates[0]
results_dir = best_pt.parent.parent
print(f"学習結果フォルダ: {results_dir}")
print(f"ベストモデル:     {best_pt}")

In [ ]:
from IPython.display import Image, display

for img_name in ['results.png', 'confusion_matrix.png', 'val_batch0_pred.jpg']:
    img_path = results_dir / img_name
    if img_path.exists():
        print(f"\n=== {img_name} ===")
        display(Image(str(img_path)))

In [ ]:
# test セットで最終評価
best_model = YOLO(str(best_pt))

test_metrics = best_model.val(
    data='data/floorplan_yolo/data.yaml',
    split='test',
    imgsz=1024,
    name='exp2_test_eval',
    project='runs/detect',
)

print("\n=== Test セット全体メトリクス (Exp 2: epochs=100) ===")
print(f"mAP@0.5:        {test_metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95:   {test_metrics.box.map:.4f}")
print(f"Precision:      {test_metrics.box.mp:.4f}")
print(f"Recall:         {test_metrics.box.mr:.4f}")

In [ ]:
# 三者比較表
import pandas as pd

class_names = ['door', 'shower', 'sink', 'staircase', 'toilet', 'window']

baseline_ap50 = {
    'door': 0.9862, 'shower': 0.4174, 'sink': 0.8231,
    'staircase': 0.7045, 'toilet': 0.9656, 'window': 0.9924
}
exp1_ap50 = {
    'door': 0.9873, 'shower': 0.7297, 'sink': 0.9617,
    'staircase': 0.7591, 'toilet': 0.9855, 'window': 0.9928
}
exp2_ap50 = {}
for i, name in enumerate(class_names):
    ap = float(test_metrics.box.ap50[i]) if i < len(test_metrics.box.ap50) else 0
    exp2_ap50[name] = ap

df = pd.DataFrame({
    'Baseline (640, 50ep)':  [baseline_ap50[c] for c in class_names],
    'Exp 1 (1024, 50ep)':    [exp1_ap50[c]     for c in class_names],
    'Exp 2 (1024, 100ep)':   [exp2_ap50[c]     for c in class_names],
}, index=class_names)
df['Δ (Exp2 - Exp1)'] = df['Exp 2 (1024, 100ep)'] - df['Exp 1 (1024, 50ep)']
df['Verdict'] = df['Δ (Exp2 - Exp1)'].apply(
    lambda x: '↑ 改善' if x > 0.01 else ('↓ 悪化' if x < -0.01 else '− 変化なし')
)

print("=" * 90)
print("クラス別 mAP@0.5: 三者比較")
print("=" * 90)
print(df.to_string(float_format=lambda x: f'{x:.4f}' if isinstance(x, float) else str(x)))

# 全体比較
baseline_overall = 0.815
exp1_overall = 0.9027
print(f"\n=== Overall mAP@0.5 三者比較 ===")
print(f"  Baseline (640,  50ep):  {baseline_overall:.4f}")
print(f"  Exp 1    (1024, 50ep):  {exp1_overall:.4f}   (vs Baseline: {exp1_overall - baseline_overall:+.4f})")
print(f"  Exp 2    (1024, 100ep): {test_metrics.box.map50:.4f}   (vs Exp 1:    {test_metrics.box.map50 - exp1_overall:+.4f})")

In [ ]:
# 推論サンプル
import random

test_images = sorted(Path('data/floorplan_yolo/test/images').glob('*.jpg'))
random.seed(42)
samples = random.sample(test_images, min(4, len(test_images)))

predict_results = best_model.predict(
    source=[str(p) for p in samples],
    imgsz=1024,
    save=True,
    project='runs/detect',
    name='exp2_samples',
    conf=0.25,
)

pred_dir = list(Path('runs').rglob('exp2_samples'))[0]
for img_path in sorted(pred_dir.glob('*.jpg')):
    print(f"\n=== {img_path.name} ===")
    display(Image(str(img_path)))

## Section 5: 結果の保存

In [ ]:
import shutil

src = str(results_dir)
out_zip = '/content/exp2_long_results.zip'
shutil.make_archive(out_zip.replace('.zip', ''), 'zip', src)
print(f"✅ Zip 作成完了: {out_zip}")
!ls -lh {out_zip}

In [ ]:
from google.colab import files
files.download(out_zip)

---

## 検証結果のまとめ(実験完了後にここに記入する)

### 仮説
imgsz=1024 のままで epochs を 50 → 100 に増やせば、shower や staircase などまだ伸びしろのあるクラスでさらなる改善が見られる。door / window はほとんど変わらない。

### 結果(↑ 上のセル出力を見て手動で記入)
- 全体 mAP@0.5: 0.903 (Exp 1) → ?
- shower mAP@0.5: 0.730 (Exp 1) → ?
- staircase mAP@0.5: 0.759 (Exp 1) → ?

### 解釈
- (改善した場合)epochs 増は精度UPに寄与する → ただし計算コストとのバランス検討
- (改善しなかった場合)Exp 1 時点で既に上限に近い → モデル容量(モデル拡大)で攻める必要あり
- (一部だけ改善した場合)クラスごとに「学習しやすさ」の天井が異なる

### 副作用
- 学習時間: 約2倍
- 過学習: train/val の loss 乖離を `results.png` で確認